# TBI Incident — Statistical Tests

This notebook runs **three statistical experiments** on the TBI incident cohort, each performed
**twice**: once at the **main-category** level (Round 1) and again at the **subcategory** level (Round 2).

| # | Experiment | Variables | Test |
|---|------------|-----------|------|
| 1 | **ICU admission rate** | `had_icu` (True/False) × incident group | Chi-Square test of independence |
| 2 | **Discharge destination** | discharge bucket (7 groups) × incident group | Chi-Square (Fisher's exact if a 2×2 cell < 5) |
| 3 | **Length of stay** | `hospital_los_days`, `icu_los_days` × incident group | Kruskal–Wallis H-test (+ Dunn's post-hoc) |

**Incident groups**

*Main categories (Round 1):* Fall, Transportation, Struck by/against, Assault, Unspecified, Multiple.

*Subcategories (Round 2):* Fall (top 17), Transportation (top 23), Struck by/against (top 8),
Assault (top 6), Unspecified (both 2), plus five specific **Multiple** combinations
(Fall + Struck by/against, Fall + Unspecified, Fall + Transportation, Fall + Late effect,
Assault + Struck by/against).

The incident classification reuses the CDC/NCHS external-cause ICD mapping from the EDA notebook
(`ICD_External_Cause_Matrix_CDC_Verified.xlsx`). There is **no `had_icu` column** in the source
data, so it is derived as `icu_los_days.notna()` — a patient counts as an ICU patient when an ICU
length-of-stay is recorded.

## 0 · Setup, data load & incident classification

In [ ]:
# !pip install scipy scikit-posthocs seaborn pandas matplotlib openpyxl
import re, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats          # chi2_contingency, fisher_exact, kruskal
import scikit_posthocs as sp     # Dunn's post-hoc test

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]       = 110
plt.rcParams["axes.titlesize"]   = 12
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", None)

# ---- one consistent colour per incident family, used in every plot ----
MAIN_COLORS = {
    "Fall":                  "#4C72B0",   # blue
    "Transportation":        "#DD8452",   # orange
    "Struck by/against":     "#55A868",   # green
    "Assault":               "#C44E52",   # red
    "Unspecified":           "#8C8C8C",   # gray
    "Late effect":           "#937860",   # brown
    "Natural/environmental": "#64B5CD",   # teal
    "Multiple":              "#8172B3",   # purple
}

# the six main incident groups tested in Round 1
MAIN_GROUP_ORDER = ["Fall", "Transportation", "Struck by/against",
                    "Assault", "Unspecified", "Multiple"]

# the seven discharge buckets (Experiment 2)
DISCH_ORDER = ["Home", "Rehab", "Skilled Nursing Facility",
               "Long-Term Care Facility", "Hospice", "Dead/Expired", "Other"]
DISCH_COLORS = dict(zip(DISCH_ORDER, sns.color_palette("Set2", len(DISCH_ORDER))))

# how many top subcategories to keep per family (Round 2)
TOP_N = {"Fall": 17, "Transportation": 23, "Struck by/against": 8,
         "Assault": 6, "Unspecified": 2}

# the five Multiple-category combinations to test as their own subgroups
MULTI_COMBOS = ["Fall + Struck by/against", "Fall + Unspecified",
                "Fall + Transportation", "Fall + Late effect",
                "Assault + Struck by/against"]

print("Libraries loaded.")

In [ ]:
PATIENT_CSV = "TBI_Incident_dataset.csv"
MATRIX_XLSX = "ICD_External_Cause_Matrix_CDC_Verified.xlsx"

df = pd.read_csv(PATIENT_CSV).drop_duplicates(subset="hadm_id").reset_index(drop=True)
print(f"Admissions: {len(df):,}   |   Unique patients: {df['subject_id'].nunique():,}")

# ============================================================
#  CDC external-cause ICD code  ->  (description, family)
#  (same logic as the EDA v3 notebook)
# ============================================================
DASH = "–"   # en-dash used inside the Excel code ranges

def family(mech):
    """Collapse a detailed CDC mechanism into one of the main families, or None."""
    m = str(mech).strip()
    if "Fall" in m:
        return "Fall"
    if (m.startswith("MVT") or m == "MV" + DASH + "Nontraffic" or m.startswith("Transport")
            or m in {"Other land transport", "Other transport",
                     "Pedestrian" + DASH + "other", "Pedal cyclist" + DASH + "other"}):
        return "Transportation"
    if m.startswith("Assault"):     return "Assault"
    if "Struck" in m:               return "Struck by/against"
    if "Late effect" in m:          return "Late effect"
    if "Natural" in m:              return "Natural/environmental"
    if m == "Unspecified":          return "Unspecified"
    return None

def expand9_full(rng):
    """All explicit ICD-9 code keys covered by an Excel range string."""
    rng = str(rng).strip()
    if rng.startswith("["):                       # 4th-digit subdivision marker, skip
        return []
    m = re.match(rf"E(\d{{3}})\.(\d){DASH}E(\d{{3}})\.(\d)$", rng)
    if m and m.group(1) == m.group(3):
        p = f"E{int(m.group(1)):03d}"
        return [f"{p}{d}" for d in range(int(m.group(2)), int(m.group(4)) + 1)]
    m = re.match(r"E(\d{3})\.(\d)$", rng)
    if m:
        return [f"E{int(m.group(1)):03d}{m.group(2)}"]
    m = re.match(rf"E(\d{{3}}){DASH}E(\d{{3}})$", rng)
    if m:
        out = []
        for n in range(int(m.group(1)), int(m.group(2)) + 1):
            p = f"E{n:03d}"
            out.append(p)
            out.extend(f"{p}{d}" for d in range(10))
        return out
    m = re.match(r"E(\d{3})$", rng)
    if m:
        p = f"E{int(m.group(1)):03d}"
        return [p] + [f"{p}{d}" for d in range(10)]
    return []

def expand10_full(rng):
    head = str(rng).split("(")[0].strip()
    m = re.match(rf"([VWXY])(\d{{2}}){DASH}([VWXY])(\d{{2}})$", head)
    if m and m.group(1) == m.group(3):
        return [f"{m.group(1)}{n:02d}" for n in range(int(m.group(2)), int(m.group(4)) + 1)]
    m = re.match(r"([VWXY]\d{2})(\.\d+)?$", head)
    if m:
        return [m.group(1)]
    return []

# ---- read both Excel mechanism sheets ----
df9 = pd.read_excel(MATRIX_XLSX, sheet_name="ICD-9 E-Codes (Mechanism)", header=3)
df9.columns = ["code_range", "description", "mechanism", "intent", "notes"]
df9 = df9.dropna(subset=["code_range", "description", "mechanism"])

df10 = pd.read_excel(MATRIX_XLSX, sheet_name="ICD-10 V-Y Codes (Mechanism)", header=3)
df10.columns = ["code_range", "description", "mechanism", "intent", "notes"]
df10 = df10.dropna(subset=["code_range", "description", "mechanism"])

# ---- build exact + (unambiguous) prefix lookups ----
EXACT9 = {}
SUB9   = defaultdict(set)
for _, r in df9.iterrows():
    fam = family(r["mechanism"])
    if fam is None:
        continue
    for c in expand9_full(r["code_range"]):
        EXACT9.setdefault(c, (r["description"], fam))
        if len(c) >= 4:
            SUB9[c[:4]].add((r["description"], fam))
PREFIX9 = {p: next(iter(s)) for p, s in SUB9.items()
           if len({fam for _, fam in s}) == 1}

EXACT10 = {}
for _, r in df10.iterrows():
    fam = family(r["mechanism"])
    if fam is None:
        continue
    for c in expand10_full(r["code_range"]):
        EXACT10.setdefault(c, (r["description"], fam))

def lookup_icd9(code):
    if code in EXACT9:                            return EXACT9[code]
    if len(code) >= 5 and code[:5] in EXACT9:     return EXACT9[code[:5]]
    if len(code) >= 4 and code[:4] in EXACT9:     return EXACT9[code[:4]]
    if len(code) >= 4 and code[:4] in PREFIX9:    return PREFIX9[code[:4]]
    return None

def lookup_icd10(code):
    return EXACT10.get(code[:3])

def classify(all_codes, version):
    """Return (set of families, list of (family, description)) for one patient."""
    main = set()
    subs = []
    if pd.isna(all_codes):
        return main, subs
    for raw in str(all_codes).split(","):
        c = raw.strip().upper()
        if not c:
            continue
        if int(version) == 9:
            if not c.startswith("E"):  continue
            hit = lookup_icd9(c)
        else:
            if c[0] not in "VWXY":     continue
            hit = lookup_icd10(c)
        if hit:
            desc, fam = hit
            main.add(fam)
            subs.append((fam, desc))
    return main, subs

_main, _subs = zip(*df.apply(lambda r: classify(r["all_icd_codes"], r["icd_version"]), axis=1))
df["main_set"]    = list(_main)
df["sub_hits"]    = list(_subs)
df["n_main_cats"] = df["main_set"].apply(len)

def assign_main(s):
    if len(s) == 0:   return None
    if len(s) == 1:   return next(iter(s))
    return "Multiple"

df["main_category"] = df["main_set"].apply(assign_main)

# ---- derive had_icu (no such column in the source data) ----
df["had_icu"] = df["icu_los_days"].notna()

# ---- discharge destination -> 7 buckets ----
def discharge_bucket(row):
    if row["hospital_expire_flag"] == 1:
        return "Dead/Expired"
    loc = row["discharge_location"]
    if pd.isna(loc):
        return np.nan
    loc = str(loc).strip().upper()
    mapping = {
        "HOME":                         "Home",
        "REHAB":                        "Rehab",
        "SKILLED NURSING FACILITY":     "Skilled Nursing Facility",
        "CHRONIC/LONG TERM ACUTE CARE": "Long-Term Care Facility",
        "HOSPICE":                      "Hospice",
        "DIED":                         "Dead/Expired",
    }
    return mapping.get(loc, "Other")

df["discharge_bucket"] = df.apply(discharge_bucket, axis=1)

n_total      = len(df)
n_classified = df["main_category"].notna().sum()
print(f"Classified with an external-cause code: {n_classified:,} of {n_total:,} "
      f"({n_classified/n_total*100:.1f}%)\n")
print("Main-category counts:")
print(df["main_category"].fillna("(none)").value_counts().to_string())

In [ ]:
# ============================================================
#  Build the two analysis frames: df_main (Round 1) and sub_long (Round 2)
# ============================================================
cat_df = df[df["main_category"].notna()].copy()

# ---- Round 1 frame: the six main incident groups, one row per patient ----
df_main = df[df["main_category"].isin(MAIN_GROUP_ORDER)].copy()

# ---- pick the top-N subcategory descriptions per family (by patient count) ----
def patient_sub_counts(main_cat):
    counts = Counter()
    for subs in cat_df["sub_hits"]:
        seen = set()
        for fam, desc in subs:
            if fam == main_cat and desc not in seen:
                counts[desc] += 1
                seen.add(desc)
    return pd.Series(counts).sort_values(ascending=False)

SUB_LABELS = {fam: list(patient_sub_counts(fam).head(n).index)
              for fam, n in TOP_N.items()}

# ordered list of every subgroup tested in Round 2 (singles first, then combos)
SUB_ORDER = []
for fam in ["Fall", "Transportation", "Struck by/against", "Assault", "Unspecified"]:
    SUB_ORDER += [f"{fam}: {d}" for d in SUB_LABELS[fam]]
SUB_ORDER += MULTI_COMBOS

# ---- Round 2 frame: one row per (patient, subgroup) membership ----
# A single-family patient with several subcodes in the same family is counted once
# PER subcategory; a Multiple patient is mapped to its sorted combo label.
PATIENT_COLS = ["had_icu", "discharge_bucket", "hospital_los_days", "icu_los_days"]
rows = []
for _, r in cat_df.iterrows():
    mc   = r["main_category"]
    base = {k: r[k] for k in PATIENT_COLS}
    if mc in TOP_N:
        seen = set()
        for fam, desc in r["sub_hits"]:
            if fam == mc and desc in SUB_LABELS[mc] and desc not in seen:
                seen.add(desc)
                rows.append({"subgroup": f"{mc}: {desc}", **base})
    elif mc == "Multiple":
        combo = " + ".join(sorted(r["main_set"]))
        if combo in MULTI_COMBOS:
            rows.append({"subgroup": combo, **base})

sub_long = pd.DataFrame(rows)

# colour every subgroup by its leading family (combos -> Multiple purple)
def subgroup_color(label):
    for fam, col in MAIN_COLORS.items():
        if label.startswith(fam + ":"):
            return col
    return MAIN_COLORS["Multiple"]

SUB_COLORS = {s: subgroup_color(s) for s in SUB_ORDER}

print("Round 1 — main-group sizes:")
print(df_main["main_category"].value_counts().reindex(MAIN_GROUP_ORDER).to_string())
print(f"\nRound 2 — {len(SUB_ORDER)} subgroups, {len(sub_long):,} (patient, subgroup) rows")
for fam in TOP_N:
    print(f"  {fam:<20}: kept {len(SUB_LABELS[fam])} subcategories")

## Experiment 1 · ICU Admission Rates — Chi-Square Test of Independence

**Question.** Is whether a patient was admitted to the ICU (`had_icu` = True / False) **independent**
of the incident group they belong to?

**Test.** Pearson's **Chi-Square test of independence** on a *group × had_icu* contingency table
(`scipy.stats.chi2_contingency`).

- **H₀ (null):** ICU admission is independent of incident group — every group has the same ICU rate.
- **H₁ (alt):** the ICU admission rate differs across at least one incident group.

**Reading the result.** A small **p-value (< 0.05)** rejects H₀: the ICU rate genuinely varies by
incident group. The **χ² statistic** grows as observed counts depart from what independence would
predict, and **degrees of freedom = (rows − 1) × (cols − 1)**. The tables below give both raw counts
and the within-group ICU **percentage**.

> *Round 2 note:* a patient with several subcategory codes in the same family is counted once per
> subcategory, so subgroup rows can sum to more than the number of distinct patients.

In [ ]:
def icu_chi2(frame, group_col, order, title):
    obs = pd.crosstab(frame[group_col], frame["had_icu"])
    obs = obs.reindex([g for g in order if g in obs.index])
    obs = obs.rename(columns={False: "No ICU", True: "ICU"})
    for col in ("No ICU", "ICU"):
        if col not in obs.columns:
            obs[col] = 0
    obs = obs[["No ICU", "ICU"]]

    chi2, p, dof, expected = stats.chi2_contingency(obs)

    table = obs.copy()
    table["Total"] = obs.sum(axis=1)
    table["ICU %"] = (obs["ICU"] / table["Total"] * 100).round(1)

    print(f"=== {title} ===")
    print(f"Chi-square = {chi2:,.2f}   |   p-value = {p:.3e}   |   dof = {dof}")
    print(f"groups = {obs.shape[0]}   |   n = {int(obs.values.sum()):,}\n")
    print("Contingency table (counts + ICU rate):")
    print(table.to_string())
    return obs, table, chi2, p, dof

### Round 1 · By main category

In [ ]:
obs1, tab1, chi2_1, p1, dof1 = icu_chi2(
    df_main, "main_category", MAIN_GROUP_ORDER,
    "Round 1 — ICU admission by main category")

rate = (obs1["ICU"] / obs1.sum(axis=1) * 100)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(rate.index, rate.values, color=[MAIN_COLORS[g] for g in rate.index])
ax.set_title("ICU admission rate by main category")
ax.set_ylabel("% admitted to ICU")
ax.set_ylim(0, rate.max() * 1.25)
ax.tick_params(axis="x", rotation=25)
for r, v in zip(bars, rate.values):
    ax.annotate(f"{v:.1f}%", (r.get_x() + r.get_width()/2, v),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

### Round 2 · By subcategory

In [ ]:
obs2, tab2, chi2_2, p2, dof2 = icu_chi2(
    sub_long, "subgroup", SUB_ORDER,
    "Round 2 — ICU admission by subcategory")

rate = (obs2["ICU"] / obs2.sum(axis=1) * 100).reindex([s for s in SUB_ORDER if s in obs2.index])
fig, ax = plt.subplots(figsize=(10, max(6, 0.32 * len(rate))))
ax.barh(rate.index[::-1], rate.values[::-1],
        color=[SUB_COLORS[s] for s in rate.index[::-1]])
ax.set_title("ICU admission rate by subcategory")
ax.set_xlabel("% admitted to ICU")
for i, v in enumerate(rate.values[::-1]):
    ax.annotate(f" {v:.1f}%", (v, i), va="center", fontsize=8)
plt.tight_layout(); plt.show()

## Experiment 2 · Discharge Destination — Chi-Square Test of Independence

We reclassify `discharge_location` into **seven** buckets and test whether the **distribution of
discharge destinations differs by incident group**.

- **H₀:** the discharge-destination distribution is the same across incident groups.
- **H₁:** at least one group has a different discharge-destination distribution.

Chi-Square is valid when expected cell counts are mostly ≥ 5. **Fisher's exact test** is used instead
**for a 2×2 table when any cell count < 5** (`scipy.stats.fisher_exact` only supports 2×2 tables).
For the larger r×c tables here an exact test is computationally impractical, so we report χ² and
**flag** how many cells fall below 5 as a caveat. Heatmaps show **observed vs. expected** counts.

### Discharge-bucket mapping

| Bucket | Original `discharge_location` (or flag) |
|--------|------------------------------------------|
| **Dead/Expired** | `hospital_expire_flag == 1`, `DIED` |
| **Home** | `HOME` |
| **Rehab** | `REHAB` |
| **Skilled Nursing Facility** | `SKILLED NURSING FACILITY` |
| **Long-Term Care Facility** | `CHRONIC/LONG TERM ACUTE CARE` |
| **Hospice** | `HOSPICE` |
| **Other** | `HOME HEALTH CARE`, `AGAINST ADVICE`, `OTHER FACILITY`, `PSYCH FACILITY`, `ACUTE HOSPITAL`, `ASSISTED LIVING`, `HEALTHCARE FACILITY` |

Rows with a **missing** `discharge_location` (and not flagged as expired) are **excluded** from this
test. The cell below confirms exactly which raw values landed in **Other**.

In [ ]:
other_raw = (df.loc[df["discharge_bucket"] == "Other", "discharge_location"]
               .fillna("(missing)").value_counts())
print("Raw discharge_location values mapped to 'Other':")
print(other_raw.to_string())

print("\nDischarge-bucket counts (whole cohort):")
print(df["discharge_bucket"].value_counts(dropna=False).reindex(DISCH_ORDER).to_string())

In [ ]:
def discharge_chi2(frame, group_col, order, title):
    sub = frame.dropna(subset=["discharge_bucket"])
    obs = pd.crosstab(sub[group_col], sub["discharge_bucket"])
    obs = obs.reindex(index=[g for g in order if g in obs.index],
                      columns=[d for d in DISCH_ORDER if d in obs.columns]).fillna(0).astype(int)

    chi2, p, dof, expected = stats.chi2_contingency(obs)
    expected = pd.DataFrame(expected, index=obs.index, columns=obs.columns)
    n_low = int((obs.values < 5).sum())

    print(f"=== {title} ===")
    print(f"Chi-square = {chi2:,.2f}   |   p-value = {p:.3e}   |   dof = {dof}")
    print(f"table = {obs.shape[0]}x{obs.shape[1]}   |   n = {int(obs.values.sum()):,}")
    print(f"observed cells < 5: {n_low} of {obs.size}")
    if obs.shape == (2, 2) and n_low > 0:
        odds, fp = stats.fisher_exact(obs.values)
        print(f"2x2 with low cells -> Fisher's exact p = {fp:.3e} (odds ratio = {odds:.3f})")
    elif n_low > 0:
        print("note: some cells < 5 -> chi-square p-value is approximate "
              "(exact test impractical for an rxc table this size).")
    print("\nObserved counts:")
    print(obs.to_string())

    h = max(3.5, 0.42 * obs.shape[0] + 1.5)
    w = max(11, 1.0 * obs.shape[1] + 7)
    fig, ax = plt.subplots(1, 2, figsize=(w, h))
    sns.heatmap(obs, annot=True, fmt="d", cmap="Blues", ax=ax[0], cbar=False,
                linewidths=.5, linecolor="white")
    ax[0].set_title(f"{title}\nObserved counts")
    sns.heatmap(expected, annot=True, fmt=".1f", cmap="Oranges", ax=ax[1], cbar=False,
                linewidths=.5, linecolor="white")
    ax[1].set_title("Expected counts (under H0)")
    for a in ax:
        a.set_xlabel("Discharge bucket"); a.set_ylabel("")
        a.tick_params(axis="x", rotation=40)
    plt.tight_layout(); plt.show()
    return obs, expected, chi2, p, dof

### Round 1 · By main category

In [ ]:
obs1d, exp1d, chi2_1d, p1d, dof1d = discharge_chi2(
    df_main, "main_category", MAIN_GROUP_ORDER,
    "Round 1 — discharge destination by main category")

### Round 2 · By subcategory

In [ ]:
obs2d, exp2d, chi2_2d, p2d, dof2d = discharge_chi2(
    sub_long, "subgroup", SUB_ORDER,
    "Round 2 — discharge destination by subcategory")

**Interpretation.** A significant χ² (**p < 0.05**) means the *mix* of discharge destinations
is not the same across incident groups — e.g. one group sends a larger share home while another goes
disproportionately to rehab, skilled nursing, or expires in hospital. Compare the **observed** heatmap
to the **expected** (independence) heatmap: cells where observed ≫ expected are destinations a group
favours, and observed ≪ expected are destinations it avoids. Where many cells fall below 5 (especially
in the subcategory round), treat the χ² p-value as approximate.

## Experiment 3 · Length of Stay — Kruskal–Wallis H-test

We compare **`hospital_los_days`** and **`icu_los_days`** across incident groups.

**Why Kruskal–Wallis instead of ANOVA?** Length-of-stay distributions are strongly **right-skewed**
— most stays are short, with a long tail of a few very long stays — so they violate the **normality**
and **equal-variance** assumptions that one-way ANOVA relies on. Kruskal–Wallis is the
**non-parametric** analogue: it ranks all observations and compares **mean ranks** (effectively
comparing medians) across groups, making it robust to skew and outliers.

- **H₀:** all groups share the same LOS distribution (equal median LOS).
- **H₁:** at least one group's LOS distribution differs.

**Dunn's post-hoc test.** Kruskal–Wallis only tells us *whether* some group differs, not *which* pairs.
When the test is significant (**p < 0.05**) we run **Dunn's test** across every pair of groups with a
**Bonferroni** correction for multiple comparisons (`scikit_posthocs.posthoc_dunn`). The output is a
matrix of **adjusted pairwise p-values**: any cell **< 0.05** marks a pair whose LOS distributions
differ significantly.

Box plots use a **log scale** for the LOS axis because of the heavy skew, and only **positive** LOS
values are kept (non-positive values are admit/discharge timestamp artefacts). For the category rounds
(≤ 8 groups) the log scale is on the **y-axis**; for the dense subcategory rounds the boxes are drawn
horizontally with the log scale on the **x-axis** so the many group labels stay readable.

In [ ]:
def kw_dunn(frame, group_col, value_col, order, title, color_map):
    d = frame[[group_col, value_col]].dropna()
    d = d[d[value_col] > 0]
    groups = [g for g in order if g in set(d[group_col])]
    samples = [d.loc[d[group_col] == g, value_col].values for g in groups]

    H, p = stats.kruskal(*samples)
    print(f"=== {title} ===")
    print(f"groups = {len(groups)}   |   n = {len(d):,}")
    print(f"Kruskal-Wallis H = {H:,.2f}   |   p-value = {p:.3e}")

    med = d.groupby(group_col)[value_col].median().reindex(groups).round(2)
    print("\nMedian LOS by group:")
    print(med.to_string())

    dunn = None
    if p < 0.05:
        dunn = sp.posthoc_dunn(d, val_col=value_col, group_col=group_col,
                               p_adjust="bonferroni").reindex(index=groups, columns=groups)
        n_sig = int((dunn.values < 0.05).sum() // 2)
        print(f"\nKruskal-Wallis significant -> Dunn's post-hoc (Bonferroni-adjusted p-values):")
        print(dunn.round(4).to_string())
        print(f"\nSignificantly different pairs (adj p < 0.05): {n_sig} of "
              f"{len(groups)*(len(groups)-1)//2}")
    else:
        print("\nKruskal-Wallis not significant (p >= 0.05) -> no post-hoc test.")

    # ---- box plot (log scale) ----
    pal = {g: color_map[g] for g in groups}
    if len(groups) <= 8:
        fig, ax = plt.subplots(figsize=(max(8, 1.3 * len(groups)), 5))
        sns.boxplot(data=d, x=group_col, y=value_col, order=groups, hue=group_col,
                    palette=pal, legend=False, showfliers=False, ax=ax)
        ax.set_yscale("log")
        ax.set_ylabel(f"{value_col} (log scale)"); ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=25)
    else:
        fig, ax = plt.subplots(figsize=(11, max(6, 0.34 * len(groups))))
        sns.boxplot(data=d, y=group_col, x=value_col, order=groups, hue=group_col,
                    palette=pal, legend=False, showfliers=False, ax=ax)
        ax.set_xscale("log")
        ax.set_xlabel(f"{value_col} (log scale)"); ax.set_ylabel("")
    ax.set_title(title)
    plt.tight_layout(); plt.show()
    return H, p, dunn

### Round 1 · By main category

In [ ]:
H_h1, p_h1, dunn_h1 = kw_dunn(
    df_main, "main_category", "hospital_los_days", MAIN_GROUP_ORDER,
    "Round 1 — Hospital LOS by main category", MAIN_COLORS)

In [ ]:
H_i1, p_i1, dunn_i1 = kw_dunn(
    df_main, "main_category", "icu_los_days", MAIN_GROUP_ORDER,
    "Round 1 — ICU LOS by main category", MAIN_COLORS)

### Round 2 · By subcategory

In [ ]:
H_h2, p_h2, dunn_h2 = kw_dunn(
    sub_long, "subgroup", "hospital_los_days", SUB_ORDER,
    "Round 2 — Hospital LOS by subcategory", SUB_COLORS)

In [ ]:
H_i2, p_i2, dunn_i2 = kw_dunn(
    sub_long, "subgroup", "icu_los_days", SUB_ORDER,
    "Round 2 — ICU LOS by subcategory", SUB_COLORS)

**How to read Dunn's results.** Each Dunn matrix is symmetric; a cell `(A, B)` holds the
Bonferroni-adjusted p-value for the hypothesis that groups *A* and *B* have the same LOS distribution.
**Values < 0.05 = a significant pairwise difference.** Cross-reference a significant pair with the
median table and box plots to see *which* group has the longer stay. Bonferroni keeps the overall
false-positive rate controlled across the many pairwise tests, so it is conservative — borderline
pairs may be flagged non-significant after correction.

**Overall takeaway.** Experiments 1–3 together describe how acute severity (ICU admission and LOS) and
disposition (discharge destination) vary across TBI mechanisms — first at a coarse category level, then
at the finer subcategory level where specific mechanisms within a family can diverge.